# Gazebo Scan Dataset Viewer

Interactive viewer for datasets recorded with **scan_data_gazebo_sim** (ROS 2 + Gazebo).

- **Scan by scan** — slider / Prev–Next buttons step through the recording
- **GT overlap** — each step shows scan *i* (raw, gray), scan *i+1* (raw, red) and scan *i* transformed by the ground-truth transform (blue). Where blue matches red, that is the ground-truth overlap.

Run with the **ml** conda environment.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from ipywidgets import IntSlider, Button, HBox, interact
from pathlib import Path
import math
import yaml

print("Imports OK")

Imports OK


## 2. Configuration

In [2]:
# Optional: set this to point directly at a dataset folder, e.g.
#   DATASET_DIR = Path.home() / "ros_ws" / "datasets" / "20260731_121737"
DATASET_DIR = "/home/tim-external/dataFolder/2D-Scan-Gazebo-Dataset"

CANDIDATE_ROOTS = [
    Path.home() / "simulation_gazebo_scan_ws" / "datasets",   # NUC default
    Path.home() / "ros_ws" / "datasets",                      # local copies (scp target)
    Path.home() / "ros_ws" / "otherPackages" / "scan_data_gazebo_sim" / "datasets",
]


def find_dataset():
    if DATASET_DIR is not None:
        d = Path(DATASET_DIR)
        if d.is_dir():
            return d
    found = []
    for root in CANDIDATE_ROOTS:
        if root.is_dir():
            found += sorted(p for p in root.iterdir() if p.is_dir())
    # Names are YYYYMMDD_HHMMSS, so lexical sort == chronological; last = newest
    return found[-1] if found else None


dataset_dir = find_dataset()
if dataset_dir is None:
    print("No dataset found! Copy one from the NUC, e.g.:")
    print("  scp -r nuc01:~/simulation_gazebo_scan_ws/datasets/20260731_121737 ~/ros_ws/datasets/")
else:
    print(f"Dataset: {dataset_dir}")

Dataset: /home/tim-external/dataFolder/2D-Scan-Gazebo-Dataset


## 3. Load dataset

In [3]:
def scan_to_points(ranges, angle_min=-math.pi, angle_max=math.pi):
    """Polar range array -> (x, y) points, filtering invalid ranges."""
    n = len(ranges)
    angles = np.linspace(angle_min, angle_max, n, endpoint=False)
    valid = np.isfinite(ranges) & (ranges > 0)
    return ranges[valid] * np.cos(angles[valid]), ranges[valid] * np.sin(angles[valid])


scans = sorted((dataset_dir / "scans").glob("*.npy"))
poses = pd.read_csv(dataset_dir / "poses.csv")
transforms_path = dataset_dir / "transforms.csv"
transforms = pd.read_csv(transforms_path) if transforms_path.is_file() else None

meta = {}
meta_path = dataset_dir / "metadata.yaml"
if meta_path.is_file():
    meta = yaml.safe_load(meta_path.read_text())

print(f"Scans: {len(scans)}  Poses: {len(poses)}  Transforms: {len(transforms) if transforms is not None else 0}")
print(f"World: {meta.get('world_name', '?')}  Beams: {meta.get('num_beams', '?')}  "
      f"Range: {meta.get('range_min_m', '?')}-{meta.get('range_max_m', '?')} m  "
      f"Rate: {meta.get('scan_update_rate_hz', '?')} Hz")

Scans: 617  Poses: 617  Transforms: 616
World: maze_small  Beams: 720  Range: 0.10000000149011612-30.0 m  Rate: 0 Hz


## 4. Interactive viewer

In [4]:
def gt_transform(i):
    """GT transform of scan i onto scan i+1: (dx, dy, dtheta)."""
    if transforms is not None:
        row = transforms[(transforms["id0"] == i) & (transforms["id1"] == i + 1)]
        if not row.empty:
            return (float(row["dx"].iloc[0]), float(row["dy"].iloc[0]),
                    float(row["dtheta"].iloc[0]))
    # Fallback: derive from consecutive GT poses
    p0, p1 = poses.iloc[i], poses.iloc[i + 1]
    return float(p1["x"] - p0["x"]), float(p1["y"] - p0["y"]), float(p1["yaw"] - p0["yaw"])


def plot_step(i=0):
    x_i, y_i = scan_to_points(np.load(scans[i]))
    x_i1, y_i1 = scan_to_points(np.load(scans[i + 1]))
    dx, dy, dth = gt_transform(i)

    # Exact frame transform: express scan i in the frame of scan i+1
    #   p' = R(-dth) p - R(yaw1)^T (dx, dy)
    c, s = math.cos(dth), math.sin(dth)
    yaw1 = poses.iloc[i + 1]["yaw"]
    c1, s1 = math.cos(yaw1), math.sin(yaw1)
    x_gt = c * x_i + s * y_i - (c1 * dx + s1 * dy)
    y_gt = -s * x_i + c * y_i + (s1 * dx - c1 * dy)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_i, y=y_i, mode="markers", marker=dict(size=2, color="gray", opacity=0.5),
        name=f"scan {i} (raw)"))
    fig.add_trace(go.Scatter(
        x=x_gt, y=y_gt, mode="markers", marker=dict(size=2, color="blue"),
        name=f"scan {i} via GT"))
    fig.add_trace(go.Scatter(
        x=x_i1, y=y_i1, mode="markers", marker=dict(size=2, color="red"),
        name=f"scan {i+1} (raw)"))
    fig.update_layout(
        title=(f"Scan {i} -> GT -> Scan {i+1}   (dx={dx:.3f} m, dy={dy:.3f} m, "
               f"d\u03b8={dth:.3f} rad, t={poses.iloc[i]['time_sec']:.1f} s)"),
        xaxis_title="x (m)", yaxis_title="y (m)",
        width=900, height=700,
        legend=dict(orientation="h", y=1.02, x=0),
    )
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    return fig


max_i = len(scans) - 2  # need scan i+1 to show an overlap
if max_i < 0:
    print("Not enough scans (need >= 2) to show an overlap.")
else:
    slider = IntSlider(min=0, max=max_i, value=0, step=1, description="Scan i",
                       continuous_update=False, readout=True, readout_format="d")
    interact(plot_step, i=slider)  # auto-displays the plot (same pattern as boreas/bremen viewers)

    prev_btn = Button(description="\u25c0 Prev")
    next_btn = Button(description="Next \u25b6")

    def on_prev(_):
        slider.value = max(slider.min, slider.value - 1)

    def on_next(_):
        slider.value = min(slider.max, slider.value + 1)

    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)

    display(HBox([prev_btn, next_btn]))

interactive(children=(IntSlider(value=0, continuous_update=False, description='Scan i', max=615), Output()), _…

## Done

Step through the recording with the slider or the Prev/Next buttons.

- **gray** = scan *i* in the robot frame
- **blue** = scan *i* transformed by the ground-truth transform
- **red** = scan *i+1* in the robot frame

Where blue overlaps red, the ground truth says the two scans see the same wall — the GT overlap.

In [5]:
print('All cells executed successfully.')

All cells executed successfully.
